In [36]:
import torch
import torch.nn as nn
import torch.profiler

def print_nested_aten_ops(func, *args,**kwargs):
    with torch.profiler.profile(
        activities=[torch.profiler.ProfilerActivity.CPU],
        record_shapes = True,
        profile_memory = True,
        with_stack=  True,
        with_flops = True,
        with_modules = True,
    ) as prof:
        func(*args, **kwargs)

    for evt in prof.events():
        # if evt.name.startswith("aten::"):
        print(f"{evt.name}")
        # Print out its stack (indent to see nesting)
        if evt.stack:
            for frame in evt.stack:
                print("   ", frame)

In [33]:
x = torch.randn(2,64)
print_nested_aten_ops(x.to,"cpu")

<FunctionEvent id=9729 name=aten::to overload_name= device_type=DeviceType.CPU node_id=-1 cpu_time=11.469us start_us=416.823 end_us=428.292 cpu_children=[] None_time=0.000us name=aten::to thread=1 input_shapes=[[2, 64], [], [], [], [], [], [], []] cpu_memory_usage=0 None_memory_usage=0 is_async=False is_remote=False seq_nr=18 is_legacy=False>



In [37]:
model = nn.Linear(64, 64)
x = torch.randn(2,64)
print("============")
print_nested_aten_ops(model, x)

aten::linear
aten::t
aten::transpose
aten::as_strided
aten::addmm
aten::expand
aten::as_strided
aten::copy_
aten::resolve_conj
aten::resolve_conj
[memory]


In [7]:
import torch
import torch.nn as nn
import torch.profiler

model = nn.Linear(4, 2)
x = torch.randn(1, 4)

with torch.profiler.profile(
    activities=[torch.profiler.ProfilerActivity.CPU],
    with_stack=True,
    record_shapes=True
) as prof:
    model(x)

prof.export_chrome_trace("trace.json")
print("Trace written to trace.json (open it in chrome://tracing)")


Trace written to trace.json (open it in chrome://tracing)


In [ ]:
class MyTensor(torch.Tensor)